In [10]:
# Step 1: Load environment variables and configure the Groq API key
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [11]:
# Step 2: Load football content from the source website
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    "https://en.wikipedia.org/wiki/Association_football"
)

loader

In [12]:
# Step 3: Download the documents from the source website
# The result is a list of LangChain Document objects.
document = loader.load()

document

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Association_football', 'title': 'Association football - Wikipedia', 'language': 'en'}, page_content='\n\n\n\nAssociation football - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nName\n\n\n\n\n\n\n\n\n2\nHistory\n\n\n\n

In [13]:
# Step 4: Split the documents into smaller overlapping chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(document)

chunks

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Association_football', 'title': 'Association football - Wikipedia', 'language': 'en'}, page_content="Association football - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nName\n\n\n\n\n\n\n\n\n2\nHistory\n\n\n\n\nToggle

In [14]:
# Step 5: Create embeddings, store them in FAISS, and run a similarity search
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize a local HuggingFace embedding model
hf_embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Build a FAISS vector store from the document chunks
from langchain_community.vectorstores import FAISS

vectorstoredb = FAISS.from_documents(
    documents=chunks,
    embedding=hf_embeddings
)

# Define the question and retrieve the three most relevant chunks
query = "What are the main rules of association football?"
results = vectorstoredb.similarity_search(query, k=3)

results[0].page_content

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7126.70it/s]


'Further information: Laws of the Game (association football)\nThere are seventeen laws in the official Laws of the Game, each containing a collection of stipulations and guidelines. The same laws are designed to apply to all levels of football for both sexes, although certain modifications for groups such as juniors, seniors and people with physical disabilities are permitted.[c] The laws are often framed in broad terms, which allow flexibility in their application depending on the nature of the game. The Laws of the Game are published by FIFA, but are maintained by the IFAB.[110] In addition to the seventeen laws, numerous IFAB decisions and other directives contribute to the regulation of association football.[111][112] Within the United States, Major League Soccer used a distinct ruleset during the 1990s,[113] and the NFHS and NCAA still use rulesets that are comparable to, but different from, the IFAB Laws.[114]\nPlayers, equipment, and officials'

In [15]:
# Step 6: Initialize the chat model used to generate the answer
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
    max_tokens=800
)

In [16]:
# Step 7: Build a document chain that answers using the retrieved context
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Define the instructions and input fields used by the language model
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant that answers questions based on the context provided."
    ),
    (
        "human",
        "{context}\n\nQuestion: {question}"
    )
])

# Create a chain that combines the retrieved documents and sends them to the model
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

# Generate an answer from the retrieved football content
document_chain.invoke({
    "context": results,
    "question": query
})

'### The 17 Laws of the Game (Association Football)\n\n| # | Law | Key Points |\n|---|-----|------------|\n| **1** | **The Field of Play** | Dimensions, markings, centre circle, corner arcs, penalty area, goal area, touch‑lines, goal‑posts, crossbar, and the ball‑in‑play area. |\n| **2** | **The Ball** | Size, weight, material, and specifications for a standard football. |\n| **3** | **Number of Players** | A team must have 11 players on the field (including a goalkeeper). Substitutions are allowed (up to 3 in most competitions, 5 in some youth/club matches). |\n| **4** | **Players’ Equipment** | Uniform (shirt, shorts, socks, shin‑guards, goalkeeper gloves), and any protective gear. |\n| **5** | **The Referee** | Single on‑field official with full authority to enforce the Laws. |\n| **6** | **The Assistant Referees** | Two side‑line officials who assist with offsides, throw‑ins, and boundary decisions. |\n| **7** | **The Fourth Official** | Manages substitutions, keeps track of time, 

In [17]:
# Step 8: Connect the retriever to the document chain
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstoredb.as_retriever(search_kwargs={"k": 3})

# Map the retrieved documents to context and pass the question through unchanged
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | document_chain
)

# Ask a new question using the complete retrieval-augmented generation chain
rag_chain.invoke("What is the offside rule?")

'**The off‑side rule (Law 11 of the Laws of the Game)**\n\n| **Key point** | **Explanation** |\n|---------------|-----------------|\n| **Definition of an off‑side position** | A player is *in an off‑side position* at the moment the ball is played to him if:  <br>• he is nearer to the opponent’s goal line than both the ball **and** the second‑last defender (usually the last outfield player), **and** <br>• he is in the opponent’s half of the pitch.  <br>Being in an off‑side position alone is **not** an offence. |\n| **When it becomes an offence** | A player in an off‑side position commits an off‑side offence **if, at the instant the ball is touched or played by a teammate, he is actively involved in play** by:  <br>• interfering with an opponent (e.g., blocking a line of sight, obstructing a defender),  <br>• gaining an advantage (e.g., receiving the ball, scoring a goal). |\n| **Exceptions – no off‑side** | A player cannot be off‑side if:  <br>• he receives the ball directly from a **th